In [ ]:
from dotenv import load_dotenv

load_dotenv()

In [21]:
from langchain.tools import tool 
from datetime import datetime, timezone, timedelta

@tool
def get_current_time(utc_offset_hours: float = 5.5) -> str:
    """Get the current date and time for a given UTC offset.

    Args:
        utc_offset_hours: Hours offset from UTC, e.g. 5.5 for IST, 0 for UTC, -5 for US Eastern.

    Returns:
        The current date and time as 'YYYY-MM-DD HH:MM:SS (UTC+HHMM)'.
    """
    tz = timezone(timedelta(hours=utc_offset_hours))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S (UTC%z)")

get_current_time.invoke({"utc_offset_hours": 0.0})
print("Name: ", get_current_time.name)
print("Description: ", get_current_time.description)
print("Args: ", get_current_time.args)

Name:  get_current_time
Description:  Get the current date and time for a given UTC offset.

Args:
    utc_offset_hours: Hours offset from UTC, e.g. 5.5 for IST, 0 for UTC, -5 for US Eastern.

Returns:
    The current date and time as 'YYYY-MM-DD HH:MM:SS (UTC+HHMM)'.
Args:  {'utc_offset_hours': {'default': 5.5, 'title': 'Utc Offset Hours', 'type': 'number'}}


In [24]:
from langchain_groq import ChatGroq

def ask_llm(question: str):
    llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=1)
    response = llm.invoke(question)
    return response.content

print(ask_llm("What is the time now ?"))

GroqError: The api_key client option must be set either by passing api_key to the client or by setting the GROQ_API_KEY environment variable

In [ ]:
from langchain.agents import create_agent

llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)

agent = create_agent(
    llm,
    tools=[get_current_time],
    system_prompt="The user is in India (IST, UTC+5:30). Always use get_current_time "
                  "with utc_offset_hours=5.5 unless they name another region. "
                  "Never ask for their time zone.",
)

result = agent.invoke({"messages": [("human", "What is the time now?")]})
for m in result["messages"]:
    m.pretty_print()

In [ ]:
# Stream a plain model reply token by token
for chunk in llm.stream("Explain decorators in 3 sentences"):
    print(chunk.content, end="", flush=True)

In [ ]:
# Stream the agent step by step (model -> tool call -> tool result -> answer)
for step in agent.stream(
    {"messages": [("human", "What is the time now?")]},
    stream_mode="updates",
):
    for node, update in step.items():
        print(f"--- {node} ---")
        update["messages"][-1].pretty_print()